In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw09.ipynb")

# Homework 09: Text Analysis of Bloomberg Articles

## Text Processing and Regular Expressions

**DATA2201 — Foundations of Data Science | Fall 2026 | 100 points**

## About The Assignment

Analyze the supplied Bloomberg articles mentioning Microsoft/MSFT from 2010–2013. Build a reproducible text-processing pipeline: canonicalize headlines, extract tokens and structured fields, construct topic features, and evaluate the limitations of regex-based analysis. These are new applications of Lec16 concepts. All graded data tasks use `data/msft_bloomberg_news.txt`; examples illustrate rules and are not substitutes for processing the full dataset.


**Submit by the deadline posted on Canvas/Gradescope.**

You must submit this assignment to Gradescope by the on-time deadline. **We strongly encourage you to submit your work to Gradescope several hours before the stated deadline.** This way, you will have ample time to reach out to staff for support if you encounter difficulties with submission. While course staff is happy to help guide you with submitting your assignment ahead of the deadline, we may not be available to assist with last-minute requests.

**Please read the instructions carefully when submitting your work to Gradescope**.

In [ ]:
# Run this cell to set up your notebook. 
import warnings
warnings.simplefilter(action="ignore")

import re
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure that pandas shows at least 280 characters in columns, so we can see full articles.
pd.set_option("max_colwidth", 280)
plt.style.use("fivethirtyeight")
sns.set()
sns.set_context("talk")

---
## Before You Start

- Complete the code and written responses in the designated answer cells. 
- Use vectorized pandas/NumPy operations. **No loops or list comprehensions unless explicitly allowed.** No function definitions are required.
- Preserve article IDs and order unless instructed otherwise. Do not hard-code answers.
- Use the supplied Bloomberg data file; do not download a replacement.

**Regex reminders:** `\b` marks a word boundary, `(?:...)` groups without capturing, and `flags=re.IGNORECASE` ignores case.

For technical help, see the [Debugging Guide](https://mtu.instructure.com/courses/1619791/pages/debugging-guide).

---
<a id="outline"></a>

## Question Navigation

- [Q1: Canonicalize headlines](#q1)
- [Q2: Tokenize and measure headlines](#q2)
- [Q3: Extract tagged publication fields](#q3)
- [Q4: Count Microsoft product mentions](#q4)
- [Q5: Extract scaled numeric quantities](#q5)
- [Q6: Identify repeated headlines](#q6)
- [Q7: Summarize yearly topic coverage](#q7)
- [Q8: Visualize topic coverage](#q8)
- [Q9: Audit regex results](#q9)

---
## Importing the Data — supplied setup

Run this cell unchanged. Each row is an article. `id` identifies the article; `title`, `released_at`, `content`, and `path` are the original fields. Empty or missing text is treated as an empty string in the supplied working Series. Do not modify `msft_news_df`, `headline_text`, `article_text`, or `release_text`.

In [ ]:
msft_news_df = pd.read_json('data/msft_bloomberg_news.txt', orient='records').set_index('id')
headline_text = msft_news_df['title'].fillna('').astype(str)
article_text = msft_news_df['content'].fillna('').astype(str)
release_text = msft_news_df['released_at'].fillna('').astype(str)
msft_news_df.head(2)

<br>

---

<a id="q1"></a>

### Question 1: Canonicalize headlines (10 points)

[Return to Question Navigation](#outline)

Create `headline_key`, a Series indexed by article ID. In order: lowercase each headline; replace `&` with `and`; replace each run of characters outside ASCII `a`–`z` and `0`–`9` with one space; strip outer spaces. Preserve digits. Name the Series `headline_key`. Do not delete words or articles.

**Example:** `  MICROSOFT & Partners: Windows-8! ` → `microsoft and partners windows 8`. Empty text becomes `''`.

**Hint:** Use `.str.lower()`, `.str.replace(..., regex=True)`, and `.str.strip()`.

In [ ]:
headline_key = ...
headline_key

In [ ]:
grader.check("q1")

<br>

---

<a id="q2"></a>

### Question 2: Tokenize and measure headlines (12 points)

[Return to Question Navigation](#outline)

Using `headline_key`, create `headline_tokens` containing all runs of one or more ASCII letters in each headline, in occurrence order. Numbers are excluded. Then create `headline_stats` DataFrame with integer columns `token_count` and `character_count`, in that order. Count all characters in the cleaned headline, including its spaces. Preserve article IDs and order.

**Example:** `windows 8 phone` gives `['windows', 'phone']`, token count `2`, and character count `15`. Empty text gives an empty list and zero counts.

**Hint:** Use `.str.findall()` and `.str.len()`.

In [ ]:
headline_tokens = ...
headline_stats = ...
headline_stats.head()

In [ ]:
grader.check("q2")

<br>

---

<a id="q3"></a>

### Question 3: Extract tagged publication fields (12 points)

[Return to Question Navigation](#outline)

Use a regex with capturing groups to extract `date_text` and `time_text` from the `<date>...</date> <time>...</time>` fields into `publication` DataFrame. Each captured value excludes the tags. Whitespace may occur between tags. Preserve the article ID index. Convert the combined text into `published_at` using the format `%B %d %Y %H:%M:%S` and `errors='coerce'`. Store that datetime column in `publication`, followed by nullable integer `year` (`Int64`). A nonmatching or invalid timestamp produces missing datetime/year values. Column order: `date_text`, `time_text`, `published_at`, `year`.

**Example:** `<date>June 7 2012</date> <time>08:04:09</time>` gives `June 7 2012`, `08:04:09`, timestamp `2012-06-07 08:04:09`, and year `2012`.

**Hint:** Negated character classes such as `[^<]+` stop at the next tag.

In [ ]:
publication = ...
publication.head()

In [ ]:
grader.check("q3")

<br>

---

<a id="q4"></a>

### Question 4: Count Microsoft product mentions (12 points)

[Return to Question Navigation](#outline)

Create `product_counts` DataFrame, with integer columns `windows`, `xbox`, and `office`, counting occurrences in each **original article body**, case-insensitively. Match these forms only: `Windows`; `Xbox` or `X-box`; `Office` or `Office 365`. Require word boundaries around each complete match. `Office 365` counts once; `officer`, `WindowsPhone`, and `XboxLive` do not match. Do not count headlines separately. A loop over the three product patterns is allowed.

**Example:** `WINDOWS, X-box and Xbox; Office 365, office, officer` gives `1, 2, 2`. These are lexical matches and do not establish product meaning in context.

**Hint:** `.str.count()` counts every match; use noncapturing groups for alternatives.

In [ ]:
product_patterns = {'windows': ..., 'xbox': ..., 'office': ...}
product_counts = ...
# A loop over product_patterns is allowed.
...
product_counts.head()

In [ ]:
grader.check("q4")

<br>

---

<a id="q5"></a>

### Question 5: Extract scaled numeric quantities (12 points)

[Return to Question Navigation](#outline)

Extract the **first** occurrence of a number followed by whitespace and `million` or `billion` from each original article body, ignoring case. Use `.str.extract()` with two capturing groups: the number and the unit. The number is an unsigned integer or decimal with digits on both sides of the period. Require a word boundary before the number and after the unit.

Create `scale_mentions` DataFrame with the same article ID index and three columns: `number` (float), `unit` (lowercase), and `value_millions` (float). Multiply a billion value by 1000; leave a million value unchanged. Keep articles with no match, using missing values in all three columns.

**Example:** `2.5 billion in revenue and 40 million users` gives `2.5`, `billion`, `2500.0`. `No quantity given` gives three missing values.

This is a substring rule, not a complete parser for signed or comma-formatted quantities. The quantities are not necessarily money.

**Hint:** Use `.str.extract()`, `.astype(float)`, `.str.lower()`, and `.map({'million': 1, 'billion': 1000})`.

In [ ]:
scale_pattern = ...
scale_mentions = ...
scale_mentions.head()

In [ ]:
grader.check("q5")

<br>

---

<a id="q6"></a>

### Question 6: Identify repeated headlines (12 points)

[Return to Question Navigation](#outline)

Use `headline_key` to count how many articles share each nonempty cleaned headline. Create `duplicate_headlines`, an integer Series indexed by `headline_key`, named `article_count`. Include only headlines appearing at least twice. Sort the index alphabetically. Exclude empty headlines. Do not delete any original articles.

**Example:** if `microsoft buys firm` appears three times, its count is `3`. A headline appearing once is excluded.

**Hint:** Filter empty strings, use `.value_counts()`, keep counts of at least two, and use `.sort_index()`.

In [ ]:
duplicate_headlines = ...
duplicate_headlines

In [ ]:
grader.check("q6")

<br>

---

<a id="q7"></a>

### Question 7: Summarize yearly topic coverage (10 points)

[Return to Question Navigation](#outline)

Create `yearly_coverage` using `publication['year']` and `product_counts`. Exclude missing years, but retain articles with zero product matches and retain duplicate headlines. Index: `year`, ascending. Columns: `n_articles`, `windows_pct`, `xbox_pct`, `office_pct`. `n_articles` is the number of articles in each year; each percentage is **100 × articles with at least one match / all articles in that year**. Multiple mentions in one article count once for coverage. Do not round.

**Example:** if a year has 10 articles and 3 mention Xbox, `xbox_pct` is `30.0`, regardless of the total number of Xbox occurrences.

**Hint:** Convert counts to booleans with `.gt(0)`, join the year, and use grouped means and sizes.

In [ ]:
yearly_coverage = ...
yearly_coverage

In [ ]:
grader.check("q7")

<!-- BEGIN QUESTION -->

<br>

---

<a id="q8"></a>

### Question 8: Visualize topic coverage (10 points)

[Return to Question Navigation](#outline)

Create one line plot from `yearly_coverage` showing each product's percentage coverage by year. Use a separate labeled line and visible markers for each product. Label axes `Publication year` and `Articles mentioning product (%)`, show every available year as an integer tick, use a descriptive title and legend, and set the y-axis to 0–100. A loop over products is allowed. Do not plot raw mention totals. No written response is required.

**Manual rubric (10):** correct yearly percentages and three lines (4); axes and title (2); legend and markers (2); integer year ticks and 0–100 scale (2).

In [ ]:
# Create your plot using yearly_coverage.
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

<a id="q9"></a>

### Question 9: Audit regex results (10 points)

[Return to Question Navigation](#outline)

Audit the real-data results in **150–200 words**.

1. Select one actual article ID containing an `office` match. Quote a short excerpt (at most 20 words) around that match, and decide whether it refers to the Microsoft product or another meaning. If no such match exists, report that and examine another matched product. State why regex alone cannot resolve every case.
2. Report the number of repeated-headline groups and the number of articles belonging to these groups (`duplicate_headlines.sum()`). Explain why identical canonical headlines do not prove identical article bodies and why repeated coverage could affect Q7.
3. Select one row from `scale_mentions` with a nonmissing `number`, identified by article ID, report its extracted value and unit, and use its surrounding text to identify what is being measured. Explain one limitation of the extraction rule. If no article has a match, report that and explain the rule's scope.

Use actual computed IDs and values; do not invent examples. Supporting inspection code is allowed without loops.

**Manual rubric (10):** real article ID/excerpt and supported interpretation (3); correct duplicate statistics and explanation (3); actual quantity/context and limitation (3); clear, appropriately qualified discussion (1).

_Type your response here._

<!-- END QUESTION -->

## Final checklist

- Run the notebook from a fresh kernel and check the available tests.
- Verify all nine answers, including the plot and the real-data audit.
- Save the notebook and follow the submission instructions below.
- Total: **100 points — 80 autograded, 20 manual**.

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Homework 09!



### Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Submit this file to the HW 09  assignment on Gradescope. 

**You are responsible for ensuring your submission follows our requirements. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out to staff prior to the deadline.


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)